# Lemmatization

Part 3 of 3 - previous: `01b-stemming.ipynb`

**What?** Mapping each word to its dictionary form (the *lemma*) using a real vocabulary (WordNet) plus morphology: `mice -> mouse`, `ran -> run`, `better -> good`.

**Why do we need it if stemming already exists?** Stemming only chops suffixes, so:
1. its output is often not a word (`studies -> studi`, `happiness -> happi`) - bad for anything a human reads (search suggestions, chatbots);
2. naive rules over-stem and merge unrelated meanings (`caring -> car`);
3. it misses irregular forms entirely - `ran`, `mice`, `was`, `better` come back unchanged, so `run` and `ran` stay two different features.

Lemmatization fixes all three because it *looks words up in a dictionary*. The cost: it is slower, and it needs to know the word's part of speech (POS) as a hint - see below.

| | Stemming | Lemmatization |
|---|---|---|
| approach | rule-based suffix chopping | dictionary + morphology lookup |
| output | may be a non-word (`studi`) | always a valid word (`study`) |
| irregulars (`ran`, `mice`, `better`) | missed | handled (`run`, `mouse`, `good`) |
| speed | fast | slower (WordNet lookup + POS) |

## Setup

One-time NLTK data download (cached in `~/nltk_data`).

In [2]:
import nltk

nltk.download("wordnet")  # the dictionary the lemmatizer looks words up in
nltk.download("omw-1.4")  # multilingual WordNet support

[nltk_data] Downloading package wordnet to /home/shubham/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/shubham/nltk_data...


True

## WordNetLemmatizer and the POS parameter

**What?** `lemmatize(word, pos=...)` returns the dictionary form *for that part of speech*. The default is `pos="n"` (noun).

**Why does POS matter?** The same spelling can be different words: `running` is its own noun (*running is good exercise*) but the verb `running` comes from `run`; `better` is only `good` when treated as an adjective. Wrong POS -> wrong (or unchanged) lemma. (NLTK's `pos_tag` can supply POS automatically - a natural next topic.)

In [3]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

for w in ["studies", "running", "better"]:
    print(w, "| as noun:", lemmatizer.lemmatize(w, pos="n"),
          "| as verb:", lemmatizer.lemmatize(w, pos="v"),
          "| as adjective:", lemmatizer.lemmatize(w, pos="a"))

studies | as noun: study | as verb: study | as adjective: studies
running | as noun: running | as verb: run | as adjective: running
better | as noun: better | as verb: better | as adjective: good


## Stemming vs lemmatization, side by side

The same word list as in `01b-stemming.ipynb` - compare the columns and watch the irregulars.

In [4]:
from nltk.stem import PorterStemmer, WordNetLemmatizer

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

words = [
    "playing", "played", "plays", "player",        # one concept, four forms
    "studies", "studying", "studied",              # same family, tricky endings
    "easily", "happiness", "happier", "unhappy",   # adverbs / adjectives
    "connected", "connection", "connections",
    "computers", "running", "runs",
    "ran", "dogs", "cats", "mice",                 # irregular forms
    "was", "were", "better", "caring",
]

print(f"{'word':<12}{'stem':<12}{'lemma (noun)':<14}{'lemma (verb)'}")
for w in words:
    print(f"{w:<12}{stemmer.stem(w):<12}{lemmatizer.lemmatize(w, pos='n'):<14}{lemmatizer.lemmatize(w, pos='v')}")

word        stem        lemma (noun)  lemma (verb)
playing     play        playing       play
played      play        played        play
plays       play        play          play
player      player      player        player
studies     studi       study         study
studying    studi       studying      study
studied     studi       studied       study
easily      easili      easily        easily
happiness   happi       happiness     happiness
happier     happier     happier       happier
unhappy     unhappi     unhappy       unhappy
connected   connect     connected     connect
connection  connect     connection    connection
connections connect     connection    connections
computers   comput      computer      computers
running     run         running       run
runs        run         run           run
ran         ran         ran           run
dogs        dog         dog           dog
cats        cat         cat           cat
mice        mice        mouse         mice
was         

Spot the differences:

- `mice -> mouse`, `ran -> run`, `better -> good`, `was -> be`: only lemmatization gets irregulars right.
- `studies -> studi` vs `study`: stems are not words, lemmas are.
- `easily`, `happiness`: already canonical words - the lemmatizer correctly leaves them alone, while stemming mangles them (`easili`, `happi`).

## Which one should you use?

- Speed at scale, search-style matching, and nobody reads the forms -> **stemming** (Snowball).
- Correct, human-readable words and irregular forms matter -> **lemmatization**.
- In practice: try both and measure - vocabulary size and the downstream metric (you will see the effect in `02-bow-ngrams-tfidf.ipynb`).